## **1. Mounting Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## **2. Importing Required Libraries**

Run the next two cells **only** when using Hash Positional Encoding. They are not required otherwise. Note that they may take a few minutes to complete.

In [ ]:
!pip install git+https://github.com/NVlabs/tiny-cuda-nn#subdirectory=bindings/torch

In [ ]:
import os
os.kill(os.getpid(), 9)  # Restart runtime
import tinycudann as tcnn

In [ ]:
import os
import numpy as np
from PIL import Image
from itertools import chain
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import Compose, ToTensor, Normalize
import cv2
import time
from torchvision.transforms.functional import crop
import imageio
from IPython.display import HTML
from base64 import b64encode
import gc
import matplotlib.pyplot as plt
import random

All of the required functions are in utils.py

In [ ]:
from google.colab import files
uploaded = files.upload()   # Opens a file picker

In [3]:
from utils import get_mgrid, jacobian, VideoFitting
from utils import PositionalEncoding, PosEncMLP, HashMLP, Siren, SineLayer
from utils import Homography, apply_homography

## **3. Functions for Train and Inference**

In [4]:
def train(path, total_steps, motion="homography", mlp_type="siren",
          verbose=True, steps_til_summary=100, first_omega_0=30, num_encoding_freqs=15, gauss_scale=5.0):
    """
    Args:
        path: video path
        total_steps: number of training steps
        motion: 'homography', 'flow', 'flow_no_tv', or 'no_motion'
        mlp_type: 'siren', 'relu_pe_nerf', 'relu_pe_fourier', 'relu_pe_hash', or 'relu_no_pe'
        first_omega_0: if mlp_type == 'siren'
        num_encoding_freqs: if mlp_type == 'relu_pe_nerf'
        gauss_scale: if mlp_type == 'relu_pe_fourier'
    Returns:
        g: motion model (or None)
        f: Scene Reconstruction model
        v.video: original video
        losses: dictionary of loss histories over steps
    """
    # Load and normalize video frames
    transform = Compose([
        ToTensor(),
        Normalize(torch.Tensor([0.5, 0.5, 0.5]), torch.Tensor([0.5, 0.5, 0.5]))
    ])
    v = VideoFitting(path, transform)
    videoloader = DataLoader(v, batch_size=1, pin_memory=True, num_workers=0)

    # Motion model selection
    if motion == "homography":
        g = Homography(hidden_features=256, hidden_layers=2).cuda()

    elif motion in ["flow", "flow_no_tv"]:
        g = Siren(in_features=3, out_features=2, hidden_features=256, hidden_layers=2, outermost_linear=True).cuda()
        # g = PosEncMLP(in_features=3, out_features=2, hidden_features=256, hidden_layers=2, num_encoding_freqs=0, encoding_type="nerf", include_input=True, outermost_linear=True).cuda()

    elif motion == "no_motion":
        g = None

    else:
        raise ValueError("motion must be 'homography', 'flow', 'flow_no_tv', or 'no_motion'")

    # Scene reconstruction model
    in_features_f = 2 if motion != "no_motion" else 3

    if mlp_type == "siren":
        f = Siren(in_features=in_features_f, out_features=3, hidden_features=256,
                   hidden_layers=3, outermost_linear=True, first_omega_0=first_omega_0, hidden_omega_0=30)

    elif mlp_type == "relu_pe_nerf":
        f = PosEncMLP(in_features=in_features_f, out_features=3, hidden_features=256,
                       hidden_layers=3, num_encoding_freqs=num_encoding_freqs, encoding_type="nerf", include_input=True, outermost_linear=True)

    elif mlp_type == "relu_pe_fourier":
        f = PosEncMLP(in_features=in_features_f, out_features=3, hidden_features=256, hidden_layers=3,
                              num_encoding_freqs=256, encoding_type="gaussian", gauss_scale=gauss_scale)

    elif mlp_type == "relu_pe_hash":
        f = HashMLP(in_features=in_features_f, out_features=3, hidden_features=256, hidden_layers=3)

    elif mlp_type == "relu_no_pe":
        f = PosEncMLP(in_features=in_features_f, out_features=3, hidden_features=256,
                       hidden_layers=3, num_encoding_freqs=0, encoding_type="nerf", include_input=True, outermost_linear=True)

    else:
        raise ValueError("mlp_type must be 'siren', 'relu_pe_fourier', 'relu_pe_nerf', or 'relu_pe_hash'")

    f = f.cuda()

    # Optimizer
    params = chain(f.parameters(), g.parameters()) if g is not None else f.parameters()
    optim = torch.optim.Adam(lr=1e-4, params=params)

    # Load single video tensor and move to GPU
    model_input, ground_truth = next(iter(videoloader))
    model_input, ground_truth = model_input[0].cuda(), ground_truth[0].cuda()

    # Subsample batch size
    batch_size = (v.H * v.W) // 8

    # Loss logs
    recon_losses = []

    for step in range(total_steps):
        start = (step * batch_size) % len(model_input)
        end = min(start + batch_size, len(model_input))

        # Extract coordinates and time
        xy = model_input[start:end, :-1].requires_grad_()
        t  = model_input[start:end, [-1]].requires_grad_()

        # Apply motion model (if any)
        if motion == "homography":
            h = g(t)
            xy_ = apply_homography(xy, h)
            model_input_f = xy_

        elif motion in ["flow", "flow_no_tv"]:
            xyt = torch.cat([xy, t], dim=-1)
            h = g(xyt)
            xy_ = xy + h
            model_input_f = xy_

        elif motion == "no_motion":
            model_input_f = model_input[start:end]

        else:
            raise ValueError

        # Predict output with scene reconstrcution model
        o = f(model_input_f)

        # Losses
        l2_loss = ((o - ground_truth[start:end]) ** 2).mean()
        l1_loss = torch.abs(o - ground_truth[start:end]).mean()
        # You can include L1 loss as well, but it does not affect performance
        loss_recon = l2_loss # + l1_loss

        # Flow regularization only if motion == "flow"
        if motion == "flow":
            loss_flow = jacobian(h, xyt).abs().mean()
            loss = loss_recon + loss_flow
        else:
            loss = loss_recon

        recon_losses.append(loss_recon.item())

        if verbose and not step % steps_til_summary:
            print("Step [%04d/%04d]: recon=%0.4f" % (step, total_steps, loss.item()))

        optim.zero_grad()
        loss.backward()
        optim.step()

    return g, f, v.video, recon_losses


In [5]:
def inference(g, f, orig, motion="homography", batch_divisor=4):
    """
    Args:
        g: Motion network
        f: Scene reconstruction network
        orig: Original video tensor [N, 3, H, W], normalized to [-1, 1]
        motion: 'homography', 'flow', or 'no_motion'
    """
    batch_size = max(1, orig.size(0) // batch_divisor)
    reconstructed = []
    original = []

    for i in range(0, orig.size(0), batch_size):
        orig_batch = orig[i:i+batch_size]

        with torch.no_grad():
            N, _, H, W = orig_batch.size()
            xyt = get_mgrid([H, W, N]).cuda()
            xy = xyt[:, :-1]
            t = xyt[:, [-1]]

            if motion == "homography":
                h = g(t)
                xy_ = apply_homography(xy, h)
                model_input_f = xy_
            elif motion == "flow":
                h = g(xyt)
                xy_ = xy + h
                model_input_f = xy_
            elif motion == "no_motion":
                model_input_f = xyt
            else:
                raise ValueError("motion must be 'homography', 'flow', or 'no_motion'")

            o_scene = f(model_input_f)
            o_scene = o_scene.view(H, W, N, 3).permute(2, 0, 1, 3).cpu().detach().numpy()
            o_scene = (np.clip(o_scene * 0.5 + 0.5, 0, 1) * 255).astype(np.uint8)

            reconstructed.extend([o_scene[j] for j in range(len(o_scene))])

            o_orig = orig_batch.permute(0, 2, 3, 1).cpu().detach().numpy()
            o_orig = (np.clip(o_orig * 0.5 + 0.5, 0, 1) * 255).astype(np.uint8)

            original.extend([o_orig[j] for j in range(len(o_orig))])

    return reconstructed, original


## **4. Applying on Experimental Capture**

In [ ]:
# ============================
# 1. Dataset Path and Setup
# ============================
# Path to the experimental video frames dataset. Update this to your local directory.
dataset_path = "/content/drive/MyDrive/1. Orange Butterfly/Experimental Capture (Frames)/" # Change to your address

# ============================
# 2. Training the Models
# ============================
# Record the start time
start_time = time.time()

# Train the motion (g) and Scene Reconstruction (f) networks
# 'mlp_type' can be 'relu_pe_fourier', 'siren', 'relu_pe_nerf', 'relu_pe_hash', or relu_no_pe
# 'first_omega_0' is for SIREN MLP ('siren')
# 'gauss_scale' is for Fourier Positional Encoding ('relu_pe_fourier')
# 'num_encoding_freqs' is for NeRF Positional Encoding ('relu_pe_nerf')
# 'motion' can be 'homography', 'flow' (with TV regularization), 'flow_no_tv', or 'no_motion'
g, f, orig, losses = train(
    dataset_path,
    total_steps=2500,       # Between 2000 to 3000 is ok.
    motion="homography",
    mlp_type="relu_pe_fourier",
    first_omega_0=30,       # if mlp_type == "siren"
    num_encoding_freqs=15,  # if mlp_type == "relu_pe_nerf" (10 or 15 are reasonable)
    gauss_scale=5.0,        # if mlp_type == "relu_pe_fourier" (Between 3 to 7 usually works well. Use 5 as default)
    verbose=True
)

# Record the end time and compute execution duration
end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.2f} seconds")
print("-" * 70)

# ============================
# 3. Model Summary and Parameters
# ============================
print_model_flag = 1  # Set to 1 to print model info

if print_model_flag:
    # Motion model summary
    if g is not None:
        print("Motion Model (g):")
        print(g)
        trainable_params_g = sum(p.numel() for p in g.parameters() if p.requires_grad)
        print(f"Number of trainable parameters (motion model): {trainable_params_g}")
        print("-" * 70)

    # Scene Reconstruction model summary
    print("Scene Reconstruction Model (f):")
    print(f)
    trainable_params_f = sum(p.numel() for p in f.parameters() if p.requires_grad)
    print(f"Number of trainable parameters (Scene Reconstruction model): {trainable_params_f}")
    print("-" * 70)

# ============================
# 4. Clear GPU Memory
# ============================
torch.cuda.empty_cache()  # Clear PyTorch CUDA cache
gc.collect()              # Run garbage collection

# ============================
# 5. Plot Training Losses
# ============================
plt.figure(figsize=(9, 5))
plt.plot(losses, color='blue')
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Training Losses Over Time")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# batch_divisor controls memory usage by splitting the input batch during inference:
#   1  -> full batch (fastest, highest memory use)
#   2–5 -> moderate split (balanced speed and memory)
#   10–20 -> small batches (slowest, safest if running out of memory)
batch_divisor = 5

# Assuming g, f trained with homography
all_o_scene, all_o_orig = inference(g, f, orig, motion="homography", batch_divisor=batch_divisor)

# File paths: 2D Captures (saved in Colab workspace)
fn_orig = f'/content/Original_Capture.mp4'
fn_scene = f'/content/Reconstruction.mp4'

# Color adjustment
all_o_scene_copy = all_o_scene.copy()
for i in range(len(all_o_scene_copy)):
    all_o_scene_copy[i] = np.clip((1.5 * all_o_scene_copy[i]), 0, 255).astype(np.uint8)

# Save videos
imageio.mimwrite(fn_orig, all_o_orig, fps=3, macro_block_size=1)
imageio.mimwrite(fn_scene, all_o_scene_copy, fps=3, macro_block_size=1)

# Display videos inline with labels
data_url_orig = "data:video/mp4;base64," + b64encode(open(fn_orig, 'rb').read()).decode()
data_url_scene = "data:video/mp4;base64," + b64encode(open(fn_scene, 'rb').read()).decode()

HTML(f'''
<div style="display: flex; gap: 20px; align-items: center;">

  <div>
    <h4 style="text-align: center;">Original Capture</h4>
    <video width=400 controls autoplay loop>
      <source src="{data_url_orig}" type="video/mp4">
    </video>
  </div>

  <div>
    <h4 style="text-align: center;">Reconstruction</h4>
    <video width=400 controls autoplay loop>
      <source src="{data_url_scene}" type="video/mp4">
    </video>
  </div>

</div>
''')

## **5. Ablation Study**

### **5.1 Ablation Study: MLP Type and Positional Encoding**

In [ ]:
# ------------------------
# Parameters (adjust if needed)
# ------------------------
mlp_type_list = ["relu_no_pe", "relu_pe_hash", "relu_pe_nerf", "siren", "relu_pe_fourier"]
dataset_path = "/content/drive/MyDrive/1. Orange Butterfly/Experimental Capture (Frames)/" # Change to your address
motion = "homography"
total_steps = 4000
batch_divisor = 5
first_omega_0 = 30
num_encoding_freqs = 15
gauss_scale = 3.0

# Base folder for saving (Colab workspace)
save_dir = "/content/MLP_Ablation"
os.makedirs(save_dir, exist_ok=True)

all_losses = {}
all_videos = {}  # Store video arrays for display

# ------------------------
# Training and inference loop
# ------------------------
for mlp_type in mlp_type_list:
    print(f"\n====== Running MLP type: {mlp_type} ======")


    start_time = time.time()

    g, f, orig, losses = train(
        dataset_path,
        total_steps=total_steps,
        motion=motion,
        mlp_type=mlp_type,
        first_omega_0=first_omega_0,
        num_encoding_freqs=num_encoding_freqs,
        gauss_scale=gauss_scale,
        verbose=True
    )

    end_time = time.time()
    print(f"Execution time: {end_time - start_time:.2f} seconds")

    # Store losses
    all_losses[mlp_type] = losses

    # Print model info
    if g is not None:
        print(g)
        print(f"Trainable params (motion model): {sum(p.numel() for p in g.parameters() if p.requires_grad)}")
    print(f"Trainable params (Scene Reconstruction model): {sum(p.numel() for p in f.parameters() if p.requires_grad)}")
    print("-"*70)

    # Clear cache
    torch.cuda.empty_cache()
    gc.collect()

    # Run inference
    all_o_scene, all_o_orig = inference(g, f, orig, motion=motion, batch_divisor=batch_divisor)
    print(f"Inference shapes: {np.shape(all_o_scene)}, {np.shape(all_o_orig)}")

    # Color adjustment and save locally in Colab workspace
    all_o_scene_copy = all_o_scene.copy()
    for i in range(len(all_o_scene_copy)):
        all_o_scene_copy[i] = np.clip((1.5 * all_o_scene_copy[i]), 0, 255).astype(np.uint8)

    fn_orig = os.path.join(save_dir, f"Original_Capture_{mlp_type}.mp4")
    fn_scene = os.path.join(save_dir, f"Reconstruction_{mlp_type}.mp4")

    imageio.mimwrite(fn_orig, all_o_orig, fps=3, macro_block_size=1)
    imageio.mimwrite(fn_scene, all_o_scene_copy, fps=3, macro_block_size=1)

    all_videos[mlp_type] = (all_o_orig, all_o_scene_copy)

# ------------------------
# Plot log10 of losses
# ------------------------
plt.figure(figsize=(9, 5))
for mlp_type, losses in all_losses.items():
    plt.plot(np.log10(losses), label=mlp_type)
plt.xlabel("Training Step")
plt.ylabel("log10(Reconstruction Loss)")
plt.title("Comparison of Reconstruction Loss Across MLP Types")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ------------------------
# Visualize resutls
# ------------------------
def display_video_row(all_videos):
    # Define the order of videos and titles
    mlp_order = ["Original", "relu_no_pe", "relu_pe_hash", "relu_pe_nerf", "siren", "relu_pe_fourier"]
    mlp_titles = ["Captured", "ReLU without PE", "ReLU with Hash PE", "ReLU with NeRF PE", "SIREN without PE", "ReLU with Fourier PE"]

    # Ensure Original is in all_videos
    if "Original" not in all_videos:
        first_mlp = next(iter(all_videos))
        all_videos["Original"] = (all_videos[first_mlp][0], all_videos[first_mlp][0])

    html_content = '<div style="display: flex; gap: 20px;">'

    for mlp_type, title in zip(mlp_order, mlp_titles):
        orig, recon = all_videos[mlp_type]

        # Show Original for "Original", reconstruction otherwise
        video_to_show = orig if mlp_type == "Original" else recon

        fn_video = f"/content/{mlp_type}.mp4"
        imageio.mimwrite(fn_video, video_to_show, fps=3)

        with open(fn_video, "rb") as f_vid:
            url_video = "data:video/mp4;base64," + b64encode(f_vid.read()).decode()

        html_content += f'''
        <div>
            <h4 style="text-align:center;">{title}</h4>
            <video width=200 controls autoplay loop>
                <source src="{url_video}" type="video/mp4">
            </video>
        </div>
        '''

    html_content += '</div>'
    display(HTML(html_content))

# Call the function
display_video_row(all_videos)


### **5.2 Ablation Study: Motion Type**

In [ ]:
# ========================
# Parameters
# ========================
motion_list = ["no_motion", "flow_no_tv", "flow", "homography"]
mlp_type_fixed = "relu_pe_fourier"
# mlp_type_fixed = "siren"


dataset_path = "/content/drive/MyDrive/1. Orange Butterfly/Experimental Capture (Frames)/" # Change to your address
save_dir = f"/content/MLP_Ablation_Motion"
os.makedirs(save_dir, exist_ok=True)

total_steps = 3000
batch_divisor = 5
first_omega_0 = 30
num_encoding_freqs = 15
steps_til_summary = 100
gauss_scale = 5.0

all_losses = {}
all_videos = {}  # store original + reconstructions for each motion

# ========================
# Training & Inference Loop
# ========================
for motion in motion_list:
    print(f"\n====== Running Motion model: {motion} (MLP={mlp_type_fixed}) ======")

    start_time = time.time()
    g, f, orig, losses = train(
        dataset_path,
        total_steps=total_steps,
        motion=motion,
        mlp_type=mlp_type_fixed,
        first_omega_0=first_omega_0,
        num_encoding_freqs=num_encoding_freqs,
        gauss_scale=gauss_scale,
        steps_til_summary=steps_til_summary,
        verbose=True
    )
    end_time = time.time()
    print(f"Execution time: {end_time - start_time:.2f} seconds")

    # Store losses
    all_losses[motion] = losses

    # Print trainable params
    if g is not None:
        print(f"Trainable params (motion model): {sum(p.numel() for p in g.parameters() if p.requires_grad)}")
    print(f"Trainable params (Scene Reconstruction model): {sum(p.numel() for p in f.parameters() if p.requires_grad)}")
    print("-"*70)

    # Clear cache
    torch.cuda.empty_cache()
    gc.collect()

    # Inference
    all_o_scene, all_o_orig = inference(
        g, f, orig, motion=motion if motion != "flow_no_tv" else "flow", batch_divisor=batch_divisor
    )
    print(f"Inference shapes for {motion}: {np.shape(all_o_scene)}, {np.shape(all_o_orig)}")

    # Color adjustment and save locally
    all_o_scene_copy = all_o_scene.copy()
    for i in range(len(all_o_scene_copy)):
        all_o_scene_copy[i] = np.clip((1.5 * all_o_scene_copy[i]), 0, 255).astype(np.uint8)

    fn_orig = os.path.join(save_dir, f'Original_Capture.mp4')
    fn_scene = os.path.join(save_dir, f'Reconstruction_{motion}.mp4')

    imageio.mimwrite(fn_orig, all_o_orig, fps=3, macro_block_size=1)
    imageio.mimwrite(fn_scene, all_o_scene_copy, fps=3, macro_block_size=1)

    all_videos[motion] = (all_o_orig, all_o_scene_copy)


# ========================
# Combined log10 reconstruction loss plot
# ========================
plt.figure(figsize=(9, 5))
colors = ['red', 'green', 'blue', 'orange'][:len(all_losses)]
for (motion, losses), c in zip(all_losses.items(), colors):
    log_loss = np.log10(np.array(losses) + 1e-8)
    plt.plot(log_loss, label=motion, linewidth=2, color=c)

plt.xlabel("Training Step", fontsize=14)
plt.ylabel("Log Reconstruction Loss (log10)", fontsize=14)
plt.title(f"Comparison of Log Reconstruction Loss Across Motion Models (MLP={mlp_type_fixed})", fontsize=16, pad=15)
plt.legend(fontsize=12, loc='upper right', frameon=True, fancybox=True)
plt.grid(alpha=0.3)
plt.tight_layout()

combined_loss_path = os.path.join(save_dir, "Combined_Log_Recon_Loss_by_Motion.png")
plt.show()



# ========================
# Visualize resutls
# ========================
def display_video_row_motion(all_videos):
    motion_order = ["Original"] + motion_list
    motion_titles = ["Captured", "No Motion", "Flow without TV Loss", "Flow without TV Loss", "Hoomography"]

    # Ensure Original exists
    if "Original" not in all_videos:
        first_motion = motion_list[0]
        all_videos["Original"] = (all_videos[first_motion][0], all_videos[first_motion][0])

    html_content = '<div style="display: flex; gap: 20px;">'

    for motion_type, title in zip(motion_order, motion_titles):
        orig, recon = all_videos[motion_type]
        video_to_show = orig if motion_type == "Original" else recon
        fn_video = f"/content/{motion_type}.mp4"
        imageio.mimwrite(fn_video, video_to_show, fps=3)

        with open(fn_video, "rb") as f_vid:
            url_video = "data:video/mp4;base64," + b64encode(f_vid.read()).decode()

        html_content += f'''
        <div>
            <h4 style="text-align:center;">{title}</h4>
            <video width=200 controls autoplay loop>
                <source src="{url_video}" type="video/mp4">
            </video>
        </div>
        '''

    html_content += '</div>'
    display(HTML(html_content))

# Call the function
display_video_row_motion(all_videos)

### **5.3 Hyperparameter Analysis of Fourier Positional Encoding**

In [ ]:
# ------------------------
# Fixed Parameters
# ------------------------
dataset_path = "/content/drive/MyDrive/1. Orange Butterfly/Experimental Capture (Frames)/" # Change to your address
motion = "homography"
mlp_type = "relu_pe_fourier"
total_steps = 3000
batch_divisor = 5
first_omega_0 = 30
num_encoding_freqs = 15
beta_list = [1, 2, 5, 10, 50]   # <-- Beta values to sweep

# ------------------------
# Save Directory
# ------------------------
save_dir = "/content/Fourier_Beta_Ablation"
os.makedirs(save_dir, exist_ok=True)

all_losses_beta = {}
all_videos_beta = {}

# ------------------------
# Training and Inference Loop
# ------------------------
for beta in beta_list:
    print(f"\n====== Running Fourier PE with beta = {beta} ======")
    start_time = time.time()

    g, f, orig, losses = train(
        dataset_path,
        total_steps=total_steps,
        motion=motion,
        mlp_type=mlp_type,
        first_omega_0=first_omega_0,
        num_encoding_freqs=num_encoding_freqs,
        gauss_scale=beta,
        verbose=True
    )

    end_time = time.time()
    print(f"Execution time (beta={beta}): {end_time - start_time:.2f} seconds")

    # Store losses
    all_losses_beta[beta] = losses

    # Print model info
    if g is not None:
        print(g)
        print(f"Trainable params (motion model): {sum(p.numel() for p in g.parameters() if p.requires_grad)}")
    print(f"Trainable params (Scene Reconstruction model): {sum(p.numel() for p in f.parameters() if p.requires_grad)}")
    print("-"*70)

    # Clear GPU cache
    torch.cuda.empty_cache()
    gc.collect()

    # ------------------------
    # Inference
    # ------------------------
    all_o_scene, all_o_orig = inference(g, f, orig, motion=motion, batch_divisor=batch_divisor)
    print(f"Inference shapes (beta={beta}): {np.shape(all_o_scene)}, {np.shape(all_o_orig)}")

    # ------------------------
    # Save Videos
    # ------------------------
    all_o_scene_copy = all_o_scene.copy()
    for i in range(len(all_o_scene_copy)):
        all_o_scene_copy[i] = np.clip((1.5 * all_o_scene_copy[i]), 0, 255).astype(np.uint8)

    fn_orig = os.path.join(save_dir, f"Original_Capture_beta{beta}.mp4")
    fn_scene = os.path.join(save_dir, f"Reconstruction_beta{beta}.mp4")

    imageio.mimwrite(fn_orig, all_o_orig, fps=3, macro_block_size=1)
    imageio.mimwrite(fn_scene, all_o_scene_copy, fps=3, macro_block_size=1)

    all_videos_beta[beta] = (all_o_orig, all_o_scene_copy)

# ------------------------
# Plot log10 of losses
# ------------------------
plt.figure(figsize=(9, 5))
for beta, losses in all_losses_beta.items():
    plt.plot(np.log10(losses), label=f"β={beta}")
plt.xlabel("Training Step")
plt.ylabel("log10(Reconstruction Loss)")
plt.title("Effect of Fourier Encoding β on Reconstruction Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ------------------------
# Visualization Function
# ------------------------
def display_video_row_beta(all_videos_beta):
    beta_order = [1, 2, 5, 10, 50]
    beta_titles = [f"β={b}" for b in beta_order]

    # Ensure Original is in videos
    if "Original" not in all_videos_beta:
        first_beta = beta_order[0]
        all_videos_beta["Original"] = (all_videos_beta[first_beta][0], all_videos_beta[first_beta][0])

    html_content = '<div style="display: flex; gap: 20px;">'
    html_content += '''
    <div>
        <h4 style="text-align:center;">Captured</h4>
        <video width=200 controls autoplay loop>
            <source src="data:video/mp4;base64,{}" type="video/mp4">
        </video>
    </div>
    '''.format(b64encode(open(os.path.join(save_dir, f"Original_Capture_beta{beta_order[0]}.mp4"), "rb").read()).decode())

    for beta, title in zip(beta_order, beta_titles):
        _, recon = all_videos_beta[beta]
        fn_video = os.path.join(save_dir, f"Reconstruction_beta{beta}.mp4")
        with open(fn_video, "rb") as f_vid:
            url_video = "data:video/mp4;base64," + b64encode(f_vid.read()).decode()

        html_content += f'''
        <div>
            <h4 style="text-align:center;">{title}</h4>
            <video width=200 controls autoplay loop>
                <source src="{url_video}" type="video/mp4">
            </video>
        </div>
        '''

    html_content += '</div>'
    display(HTML(html_content))

# ------------------------
# Display Results
# ------------------------
display_video_row_beta(all_videos_beta)
